## Feature Engineering 

In [2]:
import pandas as pd
import numpy as np

# 1. Chargement des bases nettoyées
df_matches = pd.read_csv("CSV/top14_matches_processed.csv")
df_players = pd.read_csv("CSV/top14_gold_players.csv")

df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
df_matches = df_matches.sort_values(by=['player_id', 'match_date']).reset_index(drop=True)

def generate_advanced_features(df):
    processed_list = []
    
    for player_id, group in df.groupby('player_id'):
        group = group.sort_values('match_date').copy()
        
        # Jours de repos depuis le dernier match
        group['days_since_last_match'] = group['match_date'].diff().dt.days.fillna(30)
        
        # Index temporel pour fenêtres glissantes
        group_idx = group.set_index('match_date')
        
        # 1. Charges classiques (7j, 14j, 28j, 30j, 60j)
        acute_7d = group_idx['minutes_played'].rolling('7D', closed='left').sum()
        acute_14d = group_idx['minutes_played'].rolling('14D', closed='left').sum()
        chronic_28d = group_idx['minutes_played'].rolling('28D', closed='left').sum() / 4.0
        min_30d = group_idx['minutes_played'].rolling('30D', closed='left').sum()
        min_60d = group_idx['minutes_played'].rolling('60D', closed='left').sum()
        matches_30d = group_idx['minutes_played'].rolling('30D', closed='left').count()
        
        group['acute_workload_7d'] = acute_7d.values
        group['minutes_last_14d'] = acute_14d.values
        group['chronic_workload_28d'] = chronic_28d.values
        group['minutes_last_30d'] = min_30d.values
        group['minutes_last_60d'] = min_60d.values
        group['matches_count_30d'] = matches_30d.values
        
        # Remplissage des absences de match par 0
        fill_cols = ['acute_workload_7d', 'minutes_last_14d', 'chronic_workload_28d', 
                     'minutes_last_30d', 'minutes_last_60d', 'matches_count_30d']
        group[fill_cols] = group[fill_cols].fillna(0)
        
        # ACWR Classique
        group['acwr'] = group['acute_workload_7d'] / (group['chronic_workload_28d'] + 1e-5)
        
        # 2. EWMA (Moyennes mobiles exponentielles)
        # alpha_7d = 2 / (7 + 1) = 0.25 | alpha_28d = 2 / (28 + 1) = 0.0689
        ewma_7 = group['minutes_played'].shift(1).ewm(span=7, adjust=False).mean().fillna(0)
        ewma_28 = group['minutes_played'].shift(1).ewm(span=28, adjust=False).mean().fillna(0)
        
        group['ewma_acute_7d'] = ewma_7
        group['ewma_chronic_28d'] = ewma_28
        group['acwr_ewma'] = group['ewma_acute_7d'] / (group['ewma_chronic_28d'] + 1e-5)
        
        processed_list.append(group)
        
    return pd.concat(processed_list, ignore_index=True)

print("Calcul du jeu de variables enrichi...")
df_ml_advanced = generate_advanced_features(df_matches)

# Fusion avec biométrie et création de l'IMC
df_ml_advanced = pd.merge(
    df_ml_advanced,
    df_players[['player_id', 'position', 'height_cm', 'weight_kg']],
    on='player_id',
    how='left'
)

# Calcul de l'IMC (BMI)
df_ml_advanced['bmi'] = df_ml_advanced['weight_kg'] / ((df_ml_advanced['height_cm'] / 100) ** 2)

# Mois du match (saisonnalité)
df_ml_advanced['match_month'] = pd.to_datetime(df_ml_advanced['match_date']).dt.month

# Sauvegarde
df_ml_advanced.to_csv("CSV/top14_ml_features.csv", index=False)

print("\n=== MATRICE D'ENTRAÎNEMENT ENRICHIE GÉNÉRÉE ===")
print(f"Dimensions : {df_ml_advanced.shape[0]} lignes x {df_ml_advanced.shape[1]} colonnes")
print("\nColonnes prêtes pour XGBoost :")
print(df_ml_advanced[['player_id', 'match_date', 'days_since_last_match', 'acwr', 'acwr_ewma', 'bmi', 'is_injury_return']].tail(8))

Calcul du jeu de variables enrichi...

=== MATRICE D'ENTRAÎNEMENT ENRICHIE GÉNÉRÉE ===
Dimensions : 38177 lignes x 26 colonnes

Colonnes prêtes pour XGBoost :
            player_id match_date  days_since_last_match      acwr  acwr_ewma  \
38169  zinedine-aouad 2024-02-23                   14.0  0.000000   1.008926   
38170  zinedine-aouad 2024-03-07                   13.0  0.000000   0.991026   
38171  zinedine-aouad 2024-06-29                  114.0  0.000000   0.994054   
38172  zinedine-aouad 2026-09-11                  804.0  0.000000   1.026755   
38173  zinedine-aouad 2026-09-18                    7.0  3.999985   0.864925   
38174  zinedine-aouad 2026-09-25                    7.0  1.411761   0.719751   
38175  zinedine-aouad 2026-10-09                   14.0  0.000000   0.617862   
38176  zinedine-aouad 2026-10-16                    7.0  1.391302   0.534601   

             bmi  is_injury_return  
38169  32.432432                 0  
38170  32.432432                 0  
38171  32

In [9]:
import pandas as pd

df_ml = pd.read_csv("CSV/top14_ml_features.csv")

# Filtrage sur Antoine Dupont
dupont_df = df_ml[df_ml['player_id'].str.contains('romain-Ntamack', case=False, na=False)].copy()

# Affichage des colonnes de charge clés
cols = ['player_id', 'match_date', 'days_since_last_match', 'acwr', 'acwr_ewma', 'bmi', 'is_injury_return']
display(dupont_df[cols].tail(15))

,player_id,match_date,days_since_last_match,acwr,acwr_ewma,bmi,is_injury_return
30808,romain-ntamack,2026-04-12,8.0,0.000000,0.992876,25.503616,0
30809,romain-ntamack,2026-04-18,6.0,1.624365,1.043897,25.503616,0
30810,romain-ntamack,2026-04-26,8.0,0.000000,1.077327,25.503616,0
30811,romain-ntamack,2026-05-09,13.0,0.000000,1.098294,25.503616,0
30812,romain-ntamack,2026-05-17,8.0,0.000000,1.110473,25.503616,0
30813,romain-ntamack,2026-05-30,13.0,0.000000,0.976564,25.503616,0
30814,romain-ntamack,2026-06-19,20.0,0.000000,1.026229,25.503616,0
30815,romain-ntamack,2026-06-27,8.0,0.000000,1.059270,25.503616,0
30816,romain-ntamack,2026-07-11,14.0,0.000000,1.013578,25.503616,0
30817,romain-ntamack,2026-07-18,7.0,1.981651,0.977914,25.503616,0


In [5]:
import pandas as pd
import numpy as np

# 1. Chargement de la matrice
df = pd.read_csv("CSV/top14_ml_features.csv")

print("=== 1. CONTRÔLE DE SANITÉ DE LA CIBLE ===")
returns = df[df['is_injury_return'] == 1]
normals = df[df['is_injury_return'] == 0]

print(f"Matchs normaux (0)           : {len(normals)} ({len(normals)/len(df):.2%})")
print(f"Retours de blessure (1)      : {len(returns)} ({len(returns)/len(df):.2%})")
print(f"Plage de repos observée (1)  : {returns['days_since_last_match'].min():.0f} à {returns['days_since_last_match'].max():.0f} jours")

print("\n=== 2. COMPARAISON DES MOYENNES (Sain vs Retour de blessure) ===")
cols_to_check = [
    'days_since_last_match', 'acwr', 'acwr_ewma', 
    'acute_workload_7d', 'chronic_workload_28d', 
    'minutes_last_30d', 'minutes_last_60d', 'matches_count_30d', 'bmi'
]

grouped = df.groupby('is_injury_return')[cols_to_check].mean().T
grouped.columns = ['Match Normal (0)', 'Retour Blessure (1)']
grouped['Écart (%)'] = ((grouped['Retour Blessure (1)'] - grouped['Match Normal (0)']) / (grouped['Match Normal (0)'] + 1e-5)) * 100
display(grouped.round(2))

print("\n=== 3. TOP CORRÉLATIONS AVEC IS_INJURY_RETURN ===")
num_cols = df.select_dtypes(include=[np.number]).columns
corrs = df[num_cols].corr()['is_injury_return'].drop('is_injury_return').sort_values(ascending=False)
display(corrs)

=== 1. CONTRÔLE DE SANITÉ DE LA CIBLE ===
Matchs normaux (0)           : 33774 (88.47%)
Retours de blessure (1)      : 4403 (11.53%)
Plage de repos observée (1)  : 22 à 90 jours

=== 2. COMPARAISON DES MOYENNES (Sain vs Retour de blessure) ===


,Match Normal (0),Retour Blessure (1),Écart (%)
days_since_last_match,23.89,40.57,69.82
acwr,0.95,0.00,-100.00
acwr_ewma,1.02,1.08,5.67
acute_workload_7d,21.37,0.00,-100.00
chronic_workload_28d,22.08,6.00,-72.84
minutes_last_30d,92.49,30.13,-67.42
minutes_last_60d,160.33,112.14,-30.06
matches_count_30d,1.84,0.52,-71.47
bmi,30.10,29.73,-1.24



=== 3. TOP CORRÉLATIONS AVEC IS_INJURY_RETURN ===


days_since_last_match    0.108830
ewma_acute_7d            0.096623
acwr_ewma                0.084311
ewma_chronic_28d         0.059619
height_cm                0.006576
minutes_played           0.001963
weight_kg               -0.018991
bmi                     -0.027996
match_month             -0.058145
minutes_last_60d        -0.138972
acwr                    -0.230062
acute_workload_7d       -0.234739
minutes_last_30d        -0.289348
chronic_workload_28d    -0.307839
minutes_last_14d        -0.336217
matches_count_30d       -0.372651
Name: is_injury_return, dtype: float64